In [4]:
import torch
import ase
import torch_sim as ts
from ase import Atoms
import numpy as np
from torch_sim import fire_init, fire_step, gradient_descent_init, gradient_descent_step
from torch_sim.models.fumi_tosi import   FumiTosiModel
from torch_sim.models.lennard_jones import LennardJonesModel
# Based on combination of these:
# https://pubs.aip.org/aip/jcp/article/130/4/044505/932975/Thermal-conductivity-of-molten-alkali-halides
# https://www.sciencedirect.com/science/article/pii/S2352152X22007204
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# LiCl Salt Setup
a = torch.tensor([0.421999990940094, 0.29010000824928284, 0.1581999957561493], device=torch.device(device))
c = torch.tensor([0.045570001006126404, 1.2484999895095825, 69.29000091552734], device=torch.device(device))
d = torch.tensor([0.018727000802755356, 1.4982000589370728, 139.2100067138672], device=torch.device(device))
licl_model = FumiTosiModel(
    atomic_number_zi=3,                                # Atomic number of Li
    ionic_charge_i= 1,                                 # ionic charge of Li
    ionic_charge_j= -1,                                # ionic charge of Cl
    sigma_ij= torch.tensor([1.632, 2.401, 3.170]),     # sigma_ij (in Å) for Li-Li, Li-Cl & Cl-Cl pairs
    a=a,                                               # Aij (in eV)
    b= torch.tensor([2.9200], device=torch.device(device)),                         # B (in Å^(-1))
    c=c,                                                # Cij (in eV * Å^(6))
    d=d,                                                # Dij (in eV * Å^8)
    rc=7,                                               # radial cutoff (in Å)
    device=torch.device(device),
    dtype=torch.float64,
    compute_forces=True,
    compute_stress=True,
    per_atom_energies=True,
    per_atom_stresses=True,
    use_neighbor_list=True,
)

# Create a system with 20 LiCl particles
positions = torch.randint(0, 10, (20,3)) * torch.rand(20, 3)
symbols = ['Li']*10 + ['Cl']*10
cell = [10,10,10]
atoms = Atoms(symbols=symbols, positions=positions, cell=cell, pbc=True)
licl_system = ts.initialize_state(atoms, device=torch.device(device), dtype=torch.float64)

licl_system = ts.integrate(
    system=licl_system,
    model= licl_model,
    n_steps=5000,
    temperature=600, # in kelvin
    timestep=0.00001, # in ps
    integrator=ts.Integrator.nve,
)
output = licl_model(licl_system)
# Output has the following keys:
# 'energy': total potential energy of the system, 'energies': atom-wise potential energy
# 'forcees': per-atom force, 'stress': stress on the system, 'stresses': per-atom stress.
print(output)

licl_system = ts.runners.optimize(
    system=licl_system,
    model = licl_model,
    optimizer= (fire_init, fire_step),
    convergence_fn=ts.runners.generate_energy_convergence_fn(energy_tol=1e-6),
)
preoptimization_energy = output['energy'].item()
postoptimization_energy = licl_model(licl_system)['energy'].item()
print("Did we optimize the system?", postoptimization_energy < preoptimization_energy)
print("This is preoptimization energy", preoptimization_energy)
print("This is postoptimization energy", postoptimization_energy)

{'energy': tensor([27.9324], dtype=torch.float64), 'energies': tensor([ 0.0239,  5.6636,  4.7068,  0.0170,  0.1193,  0.0290,  0.2971,  0.2784,
         0.0436,  3.5177,  0.0856,  0.9956,  2.3998,  0.0180,  0.2028,  6.6840,
         8.9998,  3.3528,  3.6000, 14.8300], dtype=torch.float64), 'forces': tensor([[ 9.3203e-03,  9.2086e-02, -6.6006e-02],
        [-1.0395e+01, -1.4705e+01, -2.5469e+01],
        [-1.1059e+01,  5.3042e+00, -1.0041e+01],
        [-2.6080e-02, -4.0751e-02,  2.9033e-02],
        [-2.3551e-01, -8.8067e-02, -3.3546e-01],
        [-2.3644e-02,  1.0706e-01,  2.6926e-02],
        [-1.3286e+00, -1.9594e-01, -2.8301e-01],
        [-2.3126e-02, -1.0675e+00,  1.2150e+00],
        [ 9.3586e-02, -3.7158e-02, -1.1709e-01],
        [-1.1137e+01, -5.4684e+00, -1.4932e+01],
        [-1.9376e-01, -7.6846e-01,  1.9930e-01],
        [ 3.3729e-01, -1.6959e+00, -5.1067e-01],
        [-1.8712e+00,  4.8682e+00,  8.6189e+00],
        [-8.6120e-02,  1.4461e-01, -3.8343e-01],
        [ 3.88